# Semaine 8 — Jour 7 : Projet de supervision et décision de livraison

Assembler qualité, performance, coût, gouvernance et sécurité dans une décision restrictive.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour le parcours complet avec entretien et challenge. J1 conserve sa durée et son contenu d’origine; il peut être réparti sur plusieurs sessions.

Prérequis : J1, Python 3.11+, JSON et tests backend. Aucun accès fournisseur, aucune clé LLM et aucune infrastructure externe nécessaires.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks : `notebooks/week08/S8_J7_supervision.ipynb` et `notebooks/week08/teacher/S8_J7_supervision_teacher.ipynb` depuis la racine du dépôt.

Critère de réussite : Fournir preuves, limites et runbook sans déclencher de déploiement.

Les observations et prix sont synthétiques; les résultats mesurent le fonctionnement du protocole local, pas un fournisseur réel.


# Objectifs mesurables

- Assembler qualité, performance, coût, gouvernance et sécurité dans une décision restrictive.
- Reproduire un refus malgré des gains de moyenne, de vitesse et de coût.
- Fournir preuves, limites et runbook sans déclencher de déploiement.

## Preuves attendues

Une sortie reproductible, une interprétation de son dénominateur et une limite opérationnelle correctement expliquée.


# S8 J7 — Projet de supervision et décision de livraison


## 1. Réunir les preuves sans les écraser
Le projet intègre cinq contrôles distincts. Quality reprend la décision et les cas critiques de J1. Performance contrôle p95 et erreurs techniques. Cost vérifie le coût par succès utile. Governance vérifie le manifeste. Security exécute la suite du contrôleur de permissions. La décision est la conjonction des cinq résultats.

Une somme pondérée peut masquer une violation de sécurité derrière un avantage financier. Le moteur garde donc un veto critique. Il affiche les contrôles qui échouent au lieu d’un score unique opaque. Une preuve absente, mal formée ou incompatible provoque une erreur de validation; elle n’est pas assimilée à un zéro rassurant ou à un contrôle non applicable.

## 2. Ce qui est réellement intégré
Les rapports d’évaluation sont recalculés depuis le dataset et les sorties de J1. Les observations synthétiques proviennent de examples/week08/fixtures. Le code vérifie la grille de trente-six essais et la cohérence du dataset, du workload et de la configuration. La grille de prix et la politique ont leurs propres preuves. La date de décision est explicite pour rendre le replay reproductible.

Les empreintes garantissent une comparaison locale de contenu. Le pipeline ne vérifie pas de signature ni d’attestation d’un environnement de collecte. Les acteurs et approbations sont fictifs. Le programme ne prétend pas être un service de gouvernance authentifié.

## 3. Deux versions, une conclusion instructive
La baseline obtient 10/12 aux cas de J1, p95=601 ms, aucun échec technique et un coût par succès utile de 0,002028 USD synthétique. Selon la politique du lab et la date du 20 octobre 2026, ses cinq contrôles passent.

Le candidat obtient 11/12, p95=461 ms et un coût utile plus faible. Son timeout conserve un maximum à 2200 ms et un taux d’erreur de 1/36, encore sous le seuil illustratif. Il reste refusé : c09 est une régression critique. Les tests du contrôleur peuvent passer en parallèle; ils ne remplacent pas le refus attendu dans les sorties de l’agent.

## 4. Codes de sortie exploitables
Le CLI produit 0 pour une décision acceptée, 1 pour une décision refusée et 2 pour une preuve invalide. Ce contrat évite de confondre refus métier et panne du moteur. Le rapport peut être écrit avec --output. Un rapport généré pour un refus est utile et doit être conservé comme preuve.

Le paquet livre une commande de validation locale. Il ne modifie aucun workflow CI existant et n’ajoute aucune action GitHub externe. Dans un pipeline, exécuter le moteur après les tests, conserver le rapport comme artefact et utiliser le code retour pour bloquer la suite. Aucun déploiement automatique n’est inclus.

## 5. Faire vivre la supervision
Le registre de version relie responsables, configuration, résultats et retour prévu. Le runbook décrit quoi examiner quand un contrôle se dégrade. Une alerte ne doit pas être supprimée en baissant simplement le seuil après observation du candidat. Toute modification de politique est une nouvelle décision, avec justification et nouvelle évaluation.

Avant un déploiement réel, ajouter un jeu réservé représentatif, une collecte contrôlée, des mesures en charge, une identité vérifiée et des essais de rollback. Un canary demande une allocation de trafic et des métriques comparables; copier le mot « canary » dans une procédure ne le met pas en œuvre. Le cours fournit le plan de revue, pas un opérateur d’infrastructure.

## 6. Présenter le projet comme ingénieur et architecte
Expliquer d’abord le problème : une meilleure moyenne a caché une régression critique. Montrer les preuves puis la décision restrictive. Décrire la frontière entre évaluation, métriques techniques, coût et permission. Terminer par une limite concrète et la prochaine expérience qui la réduirait.

Cette démarche complète vos compétences backend : contrat d’entrée strict, validation des données, codes de retour, invariants testés et traçabilité de version. Le composant IA ajoute une incertitude à mesurer; il n’efface pas les principes de conception d’un système exploitable.

## À retenir
La supervision utile relie une mesure à une décision et à une action. Une preuve critique défavorable ne disparaît ni dans la vitesse ni dans les économies.


# Lab exécutable

Cellules autonomes à partir du dépôt; aucune requête réseau.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week08/ops.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week08.demo import scenario
baseline = scenario(7, "baseline")
candidate = scenario(7, "candidate")
assert baseline["accepted"]
assert not candidate["accepted"]
assert candidate["failed_checks"] == ["quality"]
assert candidate["quality"]["rate"] > baseline["quality"]["rate"]
print({"baseline": baseline["checks"], "candidate": candidate["checks"]})

# Exercices

Conserver une première réponse avant de lire la correction.

## Exercice 1 — Expliquer le refus

Pourquoi le candidat est-il refusé alors que sa moyenne, son p95 et son coût utile sont meilleurs ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 2 — Traiter une preuve absente

Retirer une observation ou changer le dataset_hash. Quels résultats attendre ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 3 — Préparer l’exploitation

Définir l’action initiale pour une hausse des timeouts, une approbation expirée et une régression critique.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.


# Entretien

Répondre en deux minutes par question, avec un exemple et une limite.

## Question 1

Comment défendre un refus de livraison face à un gain financier ?

## Question 2

Quelle preuve manque avant une production ?


# Challenge

Produire les deux rapports complets, puis créer une variante baseline avec approbation expirée et une autre avec un plafond de coût trop bas. Vérifier des causes de refus distinctes sans changer les cas de qualité.

## Critères

Reproductibilité, garde vérifiée, distinction entre résultat mesuré et garantie non démontrée.


# Corrigé des exercices

## Exercice 1

La règle quality exige tous les cas critiques. c09 échoue. Une moyenne de 11/12 ne satisfait pas le contrat de sécurité; les autres contrôles ne compensent pas ce veto.

## Exercice 2

Une observation manquante invalide la grille; un hash différent invalide la cohérence. Le CLI retourne 2, distinct du refus valide de code 1. Aucun contrôle ne passe par défaut.

## Exercice 3

Timeouts : vérifier périmètre, dépendances et changements; approbation expirée : revue actualisée avant nouvelle décision; régression critique : bloquer et examiner le cas avant toute nouvelle livraison.


# Réponses d’entretien

## Question 1

Présenter le contrat préexistant, le cas critique et ses conséquences. Les gains restent mesurés mais ne compensent pas l’obligation violée. Documenter une solution et un nouveau test plutôt que déplacer le seuil.

## Question 2

Collecte sur système réel, corpus représentatif réservé, charge, identité/autorisation, preuve des effets et rollback testé. Les fixtures prouvent le protocole local, pas la qualité d’un modèle commercial.


# Solution du challenge

Le rapport baseline est accepté et le candidat échoue uniquement à quality. Avec valid_until au 19 octobre et date au 20 octobre, governance échoue. Avec max_usd_per_success=0.001, cost échoue. Chaque variante conserve les autres preuves et produit un evidence_hash différent.

## Vérification exécutable

Les cellules supposent que le lab élève a déjà été exécuté.

In [ ]:
from copy import deepcopy
from examples.week08.demo import load
from examples.week08.ops import assess
expired = deepcopy(load("baseline"))
expired["manifest"]["valid_until"] = "2026-10-19"
assert assess(**expired, day="2026-10-20")["failed_checks"] == ["governance"]
costly = deepcopy(load("baseline"))
costly["policy"]["max_usd_per_success"] = "0.001"
assert assess(**costly, day="2026-10-20")["failed_checks"] == ["cost"]
print("Refus qualité, gouvernance et coût isolés")

# Revue pédagogique — J7

## Acquis

- Assembler qualité, performance, coût, gouvernance et sécurité dans une décision restrictive.
- Reproduire un refus malgré des gains de moyenne, de vitesse et de coût.
- Fournir preuves, limites et runbook sans déclencher de déploiement.

## Barème indicatif sur 10

- 3 : concepts et vocabulaire correctement reliés au problème.
- 3 : reproduction du lab et lecture des résultats.
- 2 : exercices argumentés.
- 2 : challenge et limite opérationnelle.

Le parcours essentiel peut être terminé sans le challenge. Une confusion entre fixture synthétique et mesure de modèle réel doit être corrigée avant la validation du projet. Vérifier en particulier : Timeouts : vérifier périmètre, dépendances et changements; approbation expirée : revue actualisée avant nouvelle décision; régression critique : bloquer et examiner le cas avant toute nouvelle livraison.


# Références — J7

Sources de référence consultées le 20 octobre 2026. Le lab implémente ses propres contrats pédagogiques; il n’affirme pas déployer les outils cités.

- [OpenTelemetry : concepts d’observabilité](https://opentelemetry.io/docs/concepts/observability-primer/) — signaux et diagnostic.
- [OpenTelemetry : traces](https://opentelemetry.io/docs/concepts/signals/traces/) — contexte et spans pour une intégration réelle.
- [OWASP : injection de prompt, édition 2025](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) — référence datée, sans prétendre être la dernière édition.
- [OWASP : excessive agency, édition 2025](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/) — permissions et autonomie.
- [Python : Decimal](https://docs.python.org/3/library/decimal.html) — calcul décimal pour les coûts.
- [Python : unittest](https://docs.python.org/3/library/unittest.html) — reproduction des contrôles.

Lire prioritairement les sources liées au thème du jour. Les seuils, tarifs et identités du lab sont fictifs et ne proviennent pas de ces documents. Le calcul nearest-rank est défini explicitement dans le chapitre et le code.
